In [1]:
import os
import sys
import openai
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from pathlib import Path

In [11]:
from google.oauth2 import service_account
from googleapiclient.discovery import build

# Auth
SERVICE_ACCOUNT_FILE = '/Users/daianeklein/Documents/DS/job-applications-tool/h.json'
SCOPES = ["https://www.googleapis.com/auth/documents.readonly"]
creds = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)

# Docs API Service
service = build('docs', 'v1', credentials=creds)

DOCUMENT_ID = '1pXc4nsuFd5RfQFWimmKaLMxucWiV7WLZDCtP18wbCTE'

def fetch_cv_text():
    """Fetches the CV document content and extracts text."""
    doc = service.documents().get(documentId=DOCUMENT_ID).execute()

    def extract_text(document):
        text = []
        for element in document.get("body", {}).get("content", []):
            if "paragraph" in element:
                for paragraph_element in element["paragraph"]["elements"]:
                    if "textRun" in paragraph_element:
                        text.append(paragraph_element["textRun"]["content"])
        return "".join(text)

    return extract_text(doc)


if __name__ == '__main__':
    document_text = fetch_cv_text()
    print("\nDocument Content:\n", document_text)



Document Content:
 Daiane Klein
Senior Data Analyst | Analytics Engineer

São Paulo, Brazil    |   +55 11 962192070    |    Linkedin    |     Github
PROFILE SUMMARY
Data Analyst with 7+ years of experience in Data Analysis, including business and customer insights, in different industries. Proficient in Python, SQL, and Dashboard development. Strong understanding of Machine Learning, statistics, and Large Language Models (LLMs) as well as business impact and results.

 	SKILLS
	Professional Skills: 	Data Analysis | Data Science | Data Visualization | ETL | Data  Engineering
	Stacks & Tools: 	Python | SQL | Power BI |
Languages: 		Portuguese (Native) | English (C1 Advanced)

 	WORK EXPERIENCE
	AI & Data Strategy Consultant, Stealth AI Startup  (Contract)					Jan 2025 - Present
Developed an LLM-powered pipeline to analyze unstructured data and generate business insights.
Collaborated directly with customers to refine solutions, understand customer journeys and behavior, 
test hypotheses

In [63]:
load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if not OPENAI_API_KEY:
    raise ValueError('OPENAI API KEY NOT FOUND')

# Initialize OpenAI Chat Model
llm = ChatOpenAI(model_name='gpt-4o', openai_api_key=OPENAI_API_KEY)

In [52]:
doc = service.documents().get(documentId=DOCUMENT_ID).execute()
doc

{'title': 'daiane-klein-resume-template',
 'body': {'content': [{'endIndex': 1,
    'sectionBreak': {'sectionStyle': {'columnSeparatorStyle': 'NONE',
      'contentDirection': 'LEFT_TO_RIGHT',
      'sectionType': 'CONTINUOUS'}}},
   {'startIndex': 1,
    'endIndex': 14,
    'paragraph': {'elements': [{'startIndex': 1,
       'endIndex': 14,
       'textRun': {'content': 'Daiane Klein\n',
        'textStyle': {'bold': True,
         'fontSize': {'magnitude': 22, 'unit': 'PT'},
         'weightedFontFamily': {'fontFamily': 'Lato', 'weight': 400}}}}],
     'paragraphStyle': {'namedStyleType': 'NORMAL_TEXT',
      'alignment': 'CENTER',
      'direction': 'LEFT_TO_RIGHT',
      'avoidWidowAndOrphan': False}}},
   {'startIndex': 14,
    'endIndex': 55,
    'paragraph': {'elements': [{'startIndex': 14,
       'endIndex': 55,
       'textRun': {'content': 'Senior Data Analyst | Analytics Engineer\n',
        'textStyle': {'weightedFontFamily': {'fontFamily': 'Lato',
          'weight': 300}}

In [51]:
# for element in doc.get("body", {}).get("content", []):
#     print(element)


In [53]:
update_professional_skills_p = '''
The Professional skills should be: 'In love with my little italian forever'
'''

In [57]:
def fetch_professional_skills(doc:str) -> str:
    content = doc.get("body", {}).get("content", [])
    
    professional_skills = None
    paragraph_count = 0  # Track which paragraph we're processing

    for element in content:
        if "paragraph" in element:
            paragraph_count += 1  # Increment for each paragraph
            
            if paragraph_count == 9:
                professional_skills = element["paragraph"]["elements"][-1]['textRun']['content']

    return professional_skills

def get_professional_skills_llm(keywords:str, profile_summary:str) -> str:
    messages = [
        SystemMessage(content=update_professional_skills_p),
        HumanMessage(content=keywords),
        HumanMessage(content=profile_summary)
    ]

    response = llm.invoke(messages)
    return response.content.strip()

In [58]:
job_title = 'Job Title: Senior Data Analyst | Analytics Engineer'
keywords = '''Job Title Keywords: Senior Data Engineer, BI Developer
Hard Skills: Microsoft Power BI, Microsoft Azure, SQL, DAX, Azure Data Factory, Databricks, Interactive Analytics, Big Data, Cloud Environment, Tableau
Soft Skills: Problem-Solving, Collaboration, Communication, Consulting, Business Acumen
Industry-Specific Terms: KPIs, Semantic Models, Business Intelligence, Data Workloads, Workshops, Whiteboarding Sessions, Knowledge Transfer, IT Architecture
'''

In [61]:
profile_summary = fetch_professional_skills(doc)

In [64]:
get_professional_skills_llm(keywords, profile_summary)

'Based on your skills and expertise in data-related fields, here is a concise overview:\n\n---\n\n**Profile Summary:**\n\nI am a results-driven Senior Data Engineer and BI Developer proficient in leveraging advanced data technologies for impactful analytics and business intelligence solutions. With a deep understanding of Microsoft Power BI, Microsoft Azure, SQL, and DAX, I excel in building robust data pipelines and interactive analytics dashboards that drive strategic decision-making.\n\n**Professional Skills:**\n\n- **Technical Expertise:** Proficient in Azure Data Factory, Databricks, and cloud environments, ensuring efficient and scalable data solutions. Hands-on experience with Tableau for comprehensive data visualization.\n  \n- **Business Intelligence:** Skilled in developing semantic models, constructing key performance indicators (KPIs), and facilitating data-driven insights through effective business intelligence practices.\n\n- **Collaboration & Communication:** Strong abil